In [27]:
import pandas as pd 
from transformers import T5Tokenizer, Seq2SeqTrainer, T5ForConditionalGeneration, TrainingArguments 

In [28]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [29]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [30]:
train_data.shape

(14732, 3)

In [31]:
# RANDOM SAMPLING 

train_data = train_data.sample(n = 4000, random_state = 42).reset_index(drop = True)
val_data = val_data.sample(n = 500, random_state = 42).reset_index(drop = True)

In [32]:
train_data.shape

(4000, 3)

## DATA PRE-PROCESSING 

In [33]:
import re 

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # replacing lines
    text = re.sub(r"\s+", " ", text) # replacing spaces
    text = re.sub(r"<.*?>", " ", text) # replacing html tags 
    text = text.strip().lower()

    return text

In [34]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [35]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

## TOKENIZATION 

In [36]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [93]:
# RAW DATA (INPUTS) => TOKENS 

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 256, truncation = True)
    labels = tokenizer(text_target=data["summary"],max_length= 64,truncation=True)

    inputs["labels"] = labels["input_ids"]
    
    return inputs 


from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [48]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

In [49]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [50]:
len(train_dataset[0]["attention_mask"])

256

## WORKING WITH THE MODEL

In [46]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [58]:
# selecting the device 

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch

device = torch.device("cpu")

model = T5ForConditionalGeneration.from_pretrained("t5-small")
model.to(device)

print(next(model.parameters()).device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

cpu


In [78]:
# DEFINING TRAINING ARGUMENTS 

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    use_cpu=True, 
    num_train_epochs=5,
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    load_best_model_at_end=True,
    weight_decay=0.01,
    logging_steps=100,
    gradient_accumulation_steps = 4
)

In [79]:
# CREATING TRAINER => HELPS IN TRAINING CODE SIMPLIFICATION

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [80]:
# TRAINING THE MODEL

trainer.train()

Epoch,Training Loss,Validation Loss
1,6.522743,1.591695
2,6.354585,1.574849
3,6.441926,1.567537
4,6.312213,1.558931
5,6.178644,1.558330


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2500, training_loss=6.3538988525390625, metrics={'train_runtime': 4196.0046, 'train_samples_per_second': 4.766, 'train_steps_per_second': 0.596, 'total_flos': 1353418014720000.0, 'train_loss': 6.3538988525390625, 'epoch': 5.0})

In [81]:
# SAVING THE MODEL 

model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [82]:
# LOADING SAVE MODEL AND TOKENIZER 

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## TESTING THE TEXT SUMMARIZER 

In [101]:
def summarize_dialogue(dialogue):
    
    dialogue = clean_data(dialogue) # clean

    inputs = tokenizer(
        "summarize: " + dialogue,
        return_tensors="pt",
        max_length=512,
        truncation=True).to(device) # tokenize

    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"], 
        attention_mask = inputs["attention_mask"], 
        max_length = 150, 
        num_beams = 4, # generate 4 summary sequences and after comparing, return the best 
        early_stopping = True
    )

    summary = tokenizer.decode(targets[0], skip_special_tokens = True) # generating summary 
    return summary 

In [102]:
test_dialogue = """
Sarah: Hi John! How was your day?
John: Hey Sarah! It was busy but good. How about you?
Sarah: Same here! By the way, what are you doing tomorrow?
John: I'm going to visit my parents. What about you?
Sarah: I'm free tomorrow. Do you want to go shopping?
John: Sure! Where do you usually shop?
Sarah: I usually go to the local market. What time should we meet?
John: Let's meet at 10:00 a.m. near the coffee shop.
Sarah: Perfect! Oh, I almost forgot—can I borrow your charger? My phone's battery is low.
John: Yes, here it is. By the way, did you finish the project?
Sarah: Yes I did! I'll tell you all about it over coffee tomorrow. See you then!"""

summary = summarize_dialogue(test_dialogue)
print("Summary :", summary)

Summary : john is going to visit sarah's parents tomorrow at 10:00 a.m. near the coffee shop.
